In [0]:
# Bronze — Chicago Raw Ingestion (01_raw_to_bronze_chicago)
# Purpose  : Read Chicago CSV → rename columns → append to bronze.chicago_raw
# Bronze preserves raw data exactly as-is.
# History  : APPEND mode — every run gets unique _batch_id + _extract_ts

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

# Detect catalog dynamically — no hardcoded catalog name
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
if CATALOG in ("hive_metastore", "spark_catalog"):
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    project_cats = [c for c in catalogs
                    if c not in ("hive_metastore", "spark_catalog",
                                 "system", "samples", "workspace")]
    CATALOG = project_cats[0] if project_cats else CATALOG

CHI_PATH     = f"/Volumes/{CATALOG}/bronze/raw_data/Food_Inspections_20260408.csv"
BRONZE_TABLE = f"{CATALOG}.bronze.chicago_raw"

BATCH_ID         = str(uuid.uuid4())
EXTRACT_TS_STR   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
EXTRACT_DATE_STR = datetime.now().strftime("%Y-%m-%d")

print(f"Catalog      : {CATALOG}")
print(f"Batch ID     : {BATCH_ID}")
print(f"Extract time : {EXTRACT_TS_STR}")
print(f"Extract date : {EXTRACT_DATE_STR}")
print(f"Target table : {BRONZE_TABLE}")

In [0]:
files = [f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/bronze/raw_data/")]
print("Files in Volume:")
for f in files:
    print(f"  {f}")

assert "Food_Inspections_20260408.csv" in files, \
    "ERROR: Chicago CSV not found — check Volume path or filename!"
print("\n✓ Chicago file confirmed — safe to proceed")

In [0]:
# Read raw — all columns as STRING, no type casting in Bronze
# inferSchema=false is deliberate — Bronze preserves raw data exactly as-is
# Type casting happens in Silver, never here
df_chi_raw = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("multiLine",   "true")
    .option("escape",      '"')
    .option("encoding",    "UTF-8")
    .csv(CHI_PATH)
)

print(f"Row count : {df_chi_raw.count():,}")
print(f"Col count : {len(df_chi_raw.columns)}")
# Expected: 308,357 rows | 17 columns

In [0]:
# Rename all 17 columns to snake_case
# No null placeholders for Dallas columns — tables are separate
# No type casting — all stays STRING in Bronze
# Only audit metadata columns added here
df_chi = (
    df_chi_raw
    .withColumnRenamed("Inspection ID",   "inspection_id")
    .withColumnRenamed("DBA Name",        "restaurant_name")
    .withColumnRenamed("AKA Name",        "aka_name")
    .withColumnRenamed("License #",       "license_number")
    .withColumnRenamed("Facility Type",   "facility_type")
    .withColumnRenamed("Risk",            "risk_category")
    .withColumnRenamed("Address",         "street_address")
    .withColumnRenamed("City",            "city")
    .withColumnRenamed("State",           "state")
    .withColumnRenamed("Zip",             "zip_code")
    .withColumnRenamed("Inspection Date", "inspection_date")
    .withColumnRenamed("Inspection Type", "inspection_type")
    .withColumnRenamed("Results",         "inspection_result")
    .withColumnRenamed("Violations",      "violations_raw")
    .withColumnRenamed("Latitude",        "latitude")
    .withColumnRenamed("Longitude",       "longitude")
    .withColumnRenamed("Location",        "location_raw")
    # Audit metadata only — no Dallas-only placeholders
    .withColumn("city_source",   F.lit("CHI"))
    .withColumn("_batch_id",     F.lit(BATCH_ID))
    .withColumn("_extract_ts",   F.lit(EXTRACT_TS_STR))
    .withColumn("_extract_date", F.lit(EXTRACT_DATE_STR))
    .withColumn("_source_file",  F.lit("Food_Inspections_20260408.csv"))
)

# Verify all columns are Delta-safe
bad_cols = [c for c in df_chi.columns
            if any(x in c for x in [" ", "-", "(", ")", ","])]
if bad_cols:
    print(f"WARNING — bad column names: {bad_cols}")
else:
    print(f"✓ All column names are Delta-safe")
    print(f"  Total columns : {len(df_chi.columns)}")

display(df_chi.select(
    "inspection_id", "restaurant_name", "inspection_result",
    "violations_raw", "city_source", "_batch_id"
).limit(3))

In [0]:
(
    df_chi
    .write
    .format("delta")
    .mode("append")
    .partitionBy("city_source", "_extract_date")   # match existing table
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

written = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == BATCH_ID)
    .count()
)
source = df_chi.count()

print(f"✓ Written to  : {BRONZE_TABLE}")
print(f"  Batch ID    : {BATCH_ID}")
print(f"  Extract ts  : {EXTRACT_TS_STR}")
print(f"  Source rows : {source:,}")
print(f"  Written rows: {written:,}")
print(f"  Match       : {'✓' if source == written else 'MISMATCH'}")

In [0]:
# Show every batch ever loaded — proof of history preservation
# Run this notebook multiple times and each run appears as a separate row
df_batches = spark.sql(f"""
    SELECT
        city_source,
        _extract_date,
        _extract_ts,
        LEFT(_batch_id, 8)  AS batch_short,
        COUNT(*)            AS row_count
    FROM {BRONZE_TABLE}
    GROUP BY city_source, _extract_date, _extract_ts, _batch_id
    ORDER BY _extract_ts DESC
""")
display(df_batches)

In [0]:
# Delta version history — every append creates a new version
# This is the storage-level proof of history preservation
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))